[![Open In Colab](https://google.com)](https://google.com)



EXTRACT | EXTRAÇÃO

In [56]:
import pandas as pd

df = pd.read_csv('users_bank.csv', sep=';')
user_ids = df['UserID'].tolist()
print(user_ids)

[1, 2, 3, 4, 5]


TRASNFORM | TRANFORMAÇÃO

In [57]:
import json

# Usaremos o DataFrame df no lugar da API original do projeto que está em desuso.

def get_user(user_id):
  # Filtra o DataFrame para encontrar o usuário com o UserID correspondente
  user_data = df[df['UserID'] == user_id]

  if not user_data.empty:
    # Converte a linha do usuário encontrada para um dicionário
    user_dict = user_data.iloc[0].to_dict()
    # Garante que 'news' seja uma lista, mesmo que fosse NaN no CSV, para permitir adições posteriores.
    if pd.isna(user_dict.get('news')):
      user_dict['news'] = []
    return user_dict
  return None

# Percorre os user_ids e obtém os dados de cada usuário do DataFrame
users = [user for user_id in user_ids if (user := get_user(user_id)) is not None]
print(json.dumps(users, indent=2))

[
  {
    "UserID": 1,
    "name": "Maria",
    "number": "00001-1",
    "agency": 1,
    "limit": "R$ 1.000,00",
    "news": []
  },
  {
    "UserID": 2,
    "name": "Jonas",
    "number": "00002-2",
    "agency": 1,
    "limit": "R$ 1.000,00",
    "news": []
  },
  {
    "UserID": 3,
    "name": "Pedro",
    "number": "00003-3",
    "agency": 1,
    "limit": "R$ 1.000,00",
    "news": []
  },
  {
    "UserID": 4,
    "name": "Cecilia",
    "number": "00004-4",
    "agency": 1,
    "limit": "R$ 1.000,00",
    "news": []
  },
  {
    "UserID": 5,
    "name": "Carlos",
    "number": "00005-5",
    "agency": 1,
    "limit": "R$ 1.000,00",
    "news": []
  }
]


In [51]:
# Usaremos o modelo de IA GEMINI ao invés do modelo da OpenIA

!pip install google-generativeai

import google.generativeai as genai
from google.colab import userdata

# Recupera a chave da API do secrets manager cadastrada no Colab seguindo as boas práticas de segurança
GEMINI_API_KEY = userdata.get('BOOTCAMP_DIO')

genai.configure(api_key=GEMINI_API_KEY)

def generate_ai_news(user):
  # Definindo o modelo a ser utilizado (alterado para 'gemini-1.0-pro')
  model = genai.GenerativeModel('gemini-2.5-flash')

  # Estruturando o prompt para generate_content
  prompt_parts = [
      "Você é um especialista em marketing bancário voltado para financiamento de imóveis.",
      f"Crie uma mensagem para {user['name']} sobre a importância dos investimentos (máximo de 100 caracteres)"
  ]

  completion = model.generate_content(
      prompt_parts,
      generation_config={"temperature": 0.3}
  )

  return completion.text.strip('\"')

for user in users:
  news = generate_ai_news(user)
  print(news)
  user['news'].append({
      "description": news
  })

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 41.653896902s.

LOAD | CARREGAMENTO

In [ ]:
# Atualizando a coluna 'news' no DataFrame 'df'.

def update_user(user):
  for user in users:
    user_df_index = df[df['UserID'] == user['UserID']].index
    return True if user_df_index != None else False
  if not user_df_index.empty:
     df.loc[user_df_index, 'news'] = json.dumps(user['news'['description']])

for user in users:
    success = update_user(user)
    print(f"User {user['name']} updated? {success}!")

display(df)


In [54]:
import json

# Criando uma cópia do DataFrame para trabalhar e evitar SettingWithCopyWarning
df_updated = df.copy()

def update_user(user):
  for user in users:
    user_index = df_updated[df_updated['UserID'] == user['UserID']].index
    return True if user_index != None else False

    # Extraindo apenas o texto da 'description' da primeira notícia
    if not user_index.empty:
        news_description = user['news'][0]['description']
        df_updated.loc[user_index, 'news'] = news_description

    else:
        df_updated.loc[user_index, 'news'] = None

for user in users:
    success = update_user(user)
    print(f"User {user['name']} updated? {success}!")

# Atribuindo o DataFrame atualizado de volta para 'df'
df = df_updated
display(df)

User Maria updated? True!
User Jonas updated? True!
User Pedro updated? True!
User Cecilia updated? True!
User Carlos updated? True!


,UserID,name,number,agency,limit,news
0,1,Maria,00001-1,1,"R$ 1.000,00","Maria, invista no seu futuro! Sua casa é seu m..."
1,2,Jonas,00002-2,1,"R$ 1.000,00","Jonas, investir é construir seu futuro. Multip..."
2,3,Pedro,00003-3,1,"R$ 1.000,00","Pedro, invista! Seu futuro e seu imóvel são co..."
3,4,Cecilia,00004-4,1,"R$ 1.000,00","Aqui estão algumas opções, focando na conexão ..."
4,5,Carlos,00005-5,1,"R$ 1.000,00","Carlos, investir é construir seu futuro. Garan..."


Atualizando o DataFrame no arquivo CSV

In [55]:
df.to_csv('users_bank.csv', sep=';', index=False)
print("O arquivo 'users_bank.csv' foi atualizado com sucesso!")

O arquivo 'users_bank.csv' foi atualizado com sucesso!
